# 🌱 Quantum Optimisation for Financial Inclusion
### Quantum for Finance — Quantum for Humanity

This notebook applies **QAOA** to optimise the allocation of microfinance loans across underserved communities — maximising social impact while managing credit risk.

**Application:** Inspired by India's Jan Dhan Yojana, Africa's M-Pesa, and global microfinance institutions (Grameen Bank, BRAC).

**Learning source:** [IBM Quantum Learning](https://learning.quantum.ibm.com)

---

## The Problem

A microfinance institution (MFI) has a fixed lending budget $B$ and $n$ loan applicants from underserved communities. Each applicant $i$ has:
- **Social impact score** $s_i$ (income potential, family dependents, business viability)
- **Default risk** $r_i$ (credit history proxy, collateral)
- **Loan amount** $l_i$

**Objective:** Maximise total social impact subject to budget and risk constraints:

$$\max_{x \in \{0,1\}^n} \sum_i s_i x_i \quad \text{s.t.} \quad \sum_i l_i x_i \leq B, \quad \sum_i r_i x_i \leq R_{\max}$$

This is a **Quadratic Unconstrained Binary Optimisation (QUBO)** problem — perfectly suited for QAOA.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_optimization.converters import QuadraticProgramToQubo
from qiskit_algorithms import QAOA, NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import Sampler

print('✅ Imports successful')

## Step 1: Define Loan Applicants

In [ ]:
# 8 microfinance loan applicants
applicants = [
    # (name,                social_impact, default_risk, loan_amount_USD)
    ('Meera — Textile',     0.90, 0.10, 500),
    ('Rajan — Agriculture', 0.85, 0.20, 800),
    ('Fatima — Tailoring',  0.88, 0.12, 400),
    ('Kumar — Dairy Farm',  0.70, 0.35, 1200),
    ('Priya — Food Stall',  0.92, 0.08, 300),
    ('Arjun — Carpentry',   0.75, 0.22, 700),
    ('Aisha — Nursery',     0.80, 0.18, 600),
    ('Suresh — Solar',      0.95, 0.15, 900),
]

names   = [a[0] for a in applicants]
impact  = np.array([a[1] for a in applicants])
risk    = np.array([a[2] for a in applicants])
amounts = np.array([a[3] for a in applicants])

BUDGET   = 3000.0   # USD total lending budget
MAX_RISK = 1.0      # maximum total default risk score

print('Loan Applicants:')
print(f'{"Name":<25} {"Impact":>8} {"Risk":>6} {"Amount ($)":>12}')
print('-' * 55)
for a in applicants:
    print(f'{a[0]:<25} {a[1]:>8.2f} {a[2]:>6.2f} {a[3]:>12,}')
print(f'\nBudget: ${BUDGET:,.0f} | Max Risk: {MAX_RISK}')

## Step 2: Build QUBO Problem

In [ ]:
# Build Quadratic Program
qp = QuadraticProgram(name='MicrofinanceAllocation')

# Binary decision variables: x_i = 1 if applicant i receives a loan
for i, name in enumerate(names):
    qp.binary_var(name=f'x{i}')

# Objective: maximise social impact (→ minimise negative impact)
linear_obj = {f'x{i}': -impact[i] for i in range(len(applicants))}
qp.minimize(linear=linear_obj)

# Constraint 1: budget
qp.linear_constraint(
    linear={f'x{i}': amounts[i] for i in range(len(applicants))},
    sense='<=', rhs=BUDGET, name='budget'
)

# Constraint 2: maximum risk
qp.linear_constraint(
    linear={f'x{i}': risk[i] for i in range(len(applicants))},
    sense='<=', rhs=MAX_RISK, name='max_risk'
)

print(qp.export_as_lp_string())

## Step 3: Classical Optimal Solution (Baseline)

In [ ]:
exact = MinimumEigenOptimizer(NumPyMinimumEigensolver())
exact_result = exact.solve(qp)

print('\n📊 Classical Optimal Allocation:')
total_impact = 0; total_disbursed = 0; total_risk = 0
for i, x in enumerate(exact_result.x):
    if x > 0.5:
        print(f'  ✅ Fund: {names[i]:<25} | Impact: {impact[i]:.2f} | Risk: {risk[i]:.2f} | ${amounts[i]:,}')
        total_impact += impact[i]; total_disbursed += amounts[i]; total_risk += risk[i]
print(f'\n  Total Impact: {total_impact:.2f} | Disbursed: ${total_disbursed:,} / ${BUDGET:,.0f} | Risk: {total_risk:.2f}')

## Step 4: Quantum QAOA Solution

In [ ]:
# Convert to QUBO for QAOA
converter = QuadraticProgramToQubo()
qubo = converter.convert(qp)

# Run QAOA
qaoa = QAOA(sampler=Sampler(), optimizer=COBYLA(maxiter=200), reps=3)
qaoa_optimizer = MinimumEigenOptimizer(qaoa)
qaoa_result_qubo = qaoa_optimizer.solve(qubo)

# Convert back
qaoa_result = converter.interpret(qaoa_result_qubo)

print('\n⚛️  QAOA Quantum Allocation:')
q_impact = 0; q_disbursed = 0; q_risk = 0
for i, x in enumerate(qaoa_result.x):
    if x > 0.5:
        print(f'  ⚛️  Fund: {names[i]:<25} | Impact: {impact[i]:.2f} | Risk: {risk[i]:.2f} | ${amounts[i]:,}')
        q_impact += impact[i]; q_disbursed += amounts[i]; q_risk += risk[i]
print(f'\n  Total Impact: {q_impact:.2f} | Disbursed: ${q_disbursed:,} / ${BUDGET:,.0f} | Risk: {q_risk:.2f}')

## Step 5: Visualise Impact vs Risk

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
selected_q = [i for i, x in enumerate(qaoa_result.x) if x > 0.5]
not_selected = [i for i in range(len(applicants)) if i not in selected_q]

sc1 = ax.scatter(risk[not_selected], impact[not_selected], 
                 s=[amounts[i]/5 for i in not_selected],
                 c='#D1D5DB', alpha=0.7, label='Not funded', zorder=2)
sc2 = ax.scatter(risk[selected_q], impact[selected_q],
                 s=[amounts[i]/5 for i in selected_q],
                 c='#8B5CF6', alpha=0.9, label='Funded (QAOA)', zorder=3)

for i in range(len(applicants)):
    ax.annotate(names[i].split('—')[0].strip(), (risk[i], impact[i]),
                fontsize=8, ha='left', va='bottom')

ax.set_xlabel('Default Risk', fontsize=12)
ax.set_ylabel('Social Impact Score', fontsize=12)
ax.set_title('Microfinance Allocation — Quantum for Financial Inclusion\n(bubble size = loan amount)', 
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🌍 Impact at Scale

At scale, this quantum optimisation approach:
- Processes **millions of loan applications** faster than classical methods
- Removes **human bias** from lending decisions
- Maximises **social impact per dollar** deployed
- Provides **transparent, auditable** lending criteria

> *"Quantum computing can help us make financial services fair, fast, and accessible for the 1.4 billion unbanked people on Earth."*  
> — Quantum for Humanity

*Part of [Quantum for Humanity](https://github.com/vivekiniitm-stack/Quantum-for-humanity)*